# Crosscoders: Analyzing Cross-Layer Features

**Survey Reference:** Section 7.4.3 - Crosscoders: Analyzing Cross-Layer Features

## Overview

**Crosscoders** extend SAEs to learn features that are shared across multiple layers. This addresses a key limitation: individual layer SAEs may split the "same" feature into separate components at each layer.

### Why Crosscoders?

Many concepts persist across layers:
- A "Python code" feature might be active from layer 5 to layer 20
- Separate SAEs would learn 16 different "Python" features
- Crosscoders learn **one** feature with layer-specific decoders

## Learning Objectives

1. Understand cross-layer feature persistence
2. Build and train crosscoder architectures
3. Analyze how features evolve across layers
4. Compare crosscoders to per-layer SAEs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. The Feature Splitting Problem

When training separate SAEs per layer:

```
Layer 5 SAE: Feature 42 = "Python code"
Layer 6 SAE: Feature 108 = "Python code" 
Layer 7 SAE: Feature 23 = "Python code"
...
```

This makes it hard to:
- Track features across layers
- Understand feature evolution
- Build coherent circuits

## 2. Crosscoder Architecture

A crosscoder has:
- **Shared encoder:** Maps concatenated layer activations to features
- **Per-layer decoders:** Each layer has its own decoder

$$f = \text{ReLU}(W_e [x_1; x_2; ...; x_L] + b_e)$$
$$\hat{x}_l = W_d^{(l)} f + b_d^{(l)} \quad \forall l \in [1, L]$$

In [ ]:
class Crosscoder(nn.Module):
    """
    Crosscoder: Learn shared features across multiple layers.
    
    Takes activations from multiple layers and learns a single set of
    features with per-layer decoders.
    """
    
    def __init__(self, d_model: int, n_layers: int, n_features: int):
        super().__init__()
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_features = n_features
        
        # Shared encoder (from concatenated layers)
        input_dim = d_model * n_layers
        self.W_enc = nn.Parameter(torch.randn(input_dim, n_features) / np.sqrt(input_dim))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Per-layer decoders
        self.W_decs = nn.ParameterList([
            nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
            for _ in range(n_layers)
        ])
        self.b_decs = nn.ParameterList([
            nn.Parameter(torch.zeros(d_model))
            for _ in range(n_layers)
        ])
    
    def encode(self, layer_acts: list[torch.Tensor]) -> torch.Tensor:
        """
        Encode multi-layer activations to shared features.
        
        Args:
            layer_acts: List of [batch, d_model] tensors, one per layer
        """
        # Concatenate along feature dimension
        x_cat = torch.cat(layer_acts, dim=-1)  # [batch, d_model * n_layers]
        pre_acts = x_cat @ self.W_enc + self.b_enc
        return F.relu(pre_acts)
    
    def decode(self, f: torch.Tensor) -> list[torch.Tensor]:
        """Decode features to per-layer reconstructions."""
        reconstructions = []
        for l in range(self.n_layers):
            x_hat = f @ self.W_decs[l] + self.b_decs[l]
            reconstructions.append(x_hat)
        return reconstructions
    
    def forward(self, layer_acts: list[torch.Tensor]):
        """
        Full forward pass.
        
        Returns:
            reconstructions: List of per-layer reconstructions
            f: Shared feature activations
        """
        f = self.encode(layer_acts)
        reconstructions = self.decode(f)
        return reconstructions, f
    
    def get_feature_layer_contributions(self, feature_idx: int) -> torch.Tensor:
        """
        Get how a feature contributes to each layer.
        
        Returns: [n_layers, d_model] tensor of decoder columns
        """
        contributions = torch.stack([
            self.W_decs[l][feature_idx] for l in range(self.n_layers)
        ])
        return contributions

## 3. Crosscoder Loss

The loss sums reconstruction errors across all layers:

$$\mathcal{L} = \sum_{l=1}^{L} \|x_l - \hat{x}_l\|_2^2 + \lambda \|f\|_1$$

In [ ]:
def crosscoder_loss(layer_acts: list[torch.Tensor], 
                    reconstructions: list[torch.Tensor],
                    f: torch.Tensor, 
                    l1_coef: float = 1e-3) -> tuple[torch.Tensor, dict]:
    """
    Compute crosscoder loss across all layers.
    """
    # Per-layer reconstruction losses
    recon_losses = []
    for x, x_hat in zip(layer_acts, reconstructions):
        recon_losses.append((x - x_hat).pow(2).mean())
    
    total_recon = sum(recon_losses)
    l1_loss = f.abs().mean()
    total_loss = total_recon + l1_coef * l1_loss
    
    with torch.no_grad():
        # Average FVE across layers
        fves = []
        for x, x_hat in zip(layer_acts, reconstructions):
            fve = 1 - (x - x_hat).var() / x.var()
            fves.append(fve.item())
        
        l0 = (f > 0).float().sum(dim=1).mean()
    
    metrics = {
        'loss': total_loss.item(),
        'recon_loss': total_recon.item(),
        'l1_loss': l1_loss.item(),
        'avg_fve': np.mean(fves),
        'l0': l0.item(),
        'per_layer_fve': fves,
    }
    
    return total_loss, metrics

## 4. Synthetic Experiment

Create synthetic multi-layer activations with shared features.

In [ ]:
def generate_crosslayer_data(n_samples: int, d_model: int, n_layers: int,
                              n_features: int, sparsity: float = 0.9):
    """
    Generate synthetic data with features that persist across layers.
    """
    # Shared sparse feature activations
    F_true = torch.rand(n_samples, n_features) * (
        torch.rand(n_samples, n_features) > sparsity
    ).float()
    
    # Per-layer decoders (features have different effects per layer)
    layer_data = []
    for l in range(n_layers):
        W_l = torch.randn(n_features, d_model) / np.sqrt(n_features)
        # Add some layer-specific variation
        noise = torch.randn(n_samples, d_model) * 0.1
        X_l = F_true @ W_l + noise
        layer_data.append(X_l)
    
    return layer_data, F_true

# Generate data
d_model = 64
n_layers = 4
n_features = 128
n_samples = 10000

layer_data, F_true = generate_crosslayer_data(n_samples, d_model, n_layers, n_features)
print(f"Layer shapes: {[x.shape for x in layer_data]}")

In [ ]:
def train_crosscoder(crosscoder, layer_data, n_epochs=100, batch_size=256,
                     lr=1e-3, l1_coef=1e-3):
    """Train a crosscoder on multi-layer data."""
    optimizer = torch.optim.Adam(crosscoder.parameters(), lr=lr)
    history = []
    n_samples = layer_data[0].size(0)
    
    for epoch in tqdm(range(n_epochs)):
        perm = torch.randperm(n_samples)
        epoch_metrics = []
        
        for i in range(0, n_samples, batch_size):
            batch_layers = [x[perm[i:i+batch_size]].to(device) for x in layer_data]
            
            optimizer.zero_grad()
            reconstructions, f = crosscoder(batch_layers)
            loss, metrics = crosscoder_loss(batch_layers, reconstructions, f, l1_coef)
            loss.backward()
            optimizer.step()
            
            epoch_metrics.append(metrics)
        
        # Average metrics (excluding per_layer_fve which is a list)
        avg_metrics = {
            k: np.mean([m[k] for m in epoch_metrics]) 
            for k in ['loss', 'recon_loss', 'l1_loss', 'avg_fve', 'l0']
        }
        history.append(avg_metrics)
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss={avg_metrics['loss']:.4f}, " +
                  f"FVE={avg_metrics['avg_fve']:.4f}, L0={avg_metrics['l0']:.1f}")
    
    return history

# Train crosscoder
crosscoder = Crosscoder(d_model, n_layers, n_features=256).to(device)
history = train_crosscoder(crosscoder, layer_data, n_epochs=100, l1_coef=5e-3)

## 5. Visualizing Cross-Layer Features

See how each feature's decoder varies across layers.

In [ ]:
def visualize_crosslayer_feature(crosscoder, feature_idx):
    """Visualize how a feature contributes to each layer."""
    contributions = crosscoder.get_feature_layer_contributions(feature_idx)
    contributions = contributions.detach().cpu().numpy()
    
    # Plot heatmap
    plt.figure(figsize=(12, 4))
    plt.imshow(contributions, aspect='auto', cmap='RdBu_r')
    plt.colorbar(label='Decoder weight')
    plt.xlabel('Dimension')
    plt.ylabel('Layer')
    plt.title(f'Feature {feature_idx}: Contribution per Layer')
    plt.show()
    
    # Plot norms
    norms = np.linalg.norm(contributions, axis=1)
    plt.figure(figsize=(8, 4))
    plt.bar(range(len(norms)), norms)
    plt.xlabel('Layer')
    plt.ylabel('Decoder Norm')
    plt.title(f'Feature {feature_idx}: Strength per Layer')
    plt.show()

# TODO: Visualize specific features
# visualize_crosslayer_feature(crosscoder, 0)

## 6. Comparison: Per-Layer SAEs vs. Crosscoder

| Aspect | Per-Layer SAEs | Crosscoder |
|--------|---------------|------------|
| Features | Different per layer | Shared across layers |
| Tracking | Requires feature matching | Built-in |
| Efficiency | L × M features | M features |
| Evolution | Hard to see | Clear from decoders |

In [ ]:
# TODO: Compare feature quality between approaches

## Key Takeaways

1. **Crosscoders unify features** across layers into a single representation
2. **Per-layer decoders** show how each feature contributes to each layer
3. **Better for circuit analysis** since features are coherent across depth
4. **Trade-off:** May miss layer-specific features that don't generalize
5. **Useful for:** Understanding how information flows through the network

## Next Steps

→ **04_circuits/01_circuit_discovery.ipynb**: Use features to discover computational circuits